In [ ]:
######################## 
# IMPORTS #

import pygame, sys
from pygame.locals import *
import numpy as np
import tkinter as tk

In [ ]:
######################## 
# INPUTS #
# Parity check matrix to be drawn. numpy array is standard, should be binary
parityCheckMatrix = np.ones((4,8))
# List of strings containing labels for each check node (ordered as columns in H)
cheLabels = ['1,3,4,2','G_2222das','abe','0']
# List of strings containing labels for each variable node (ordered as rows in H)
varLabels = ['1.0','33.','arg','a','s','d','f','1']

######################## 
# INITIALIZATION #
pygame.init()

######################## 
# COLORS #
BLUE  = (0, 0, 255)
RED   = (255, 0, 0)
GREEN = (0, 255, 0)
BLACK = (0, 0, 0)
WHITE = (255, 255, 255)

######################## 
# FONTS #
font_small = pygame.font.SysFont("Verdana", 12)

######################## 
# FPS #
FPS = 30
FramePerSec = pygame.time.Clock()

######################## 
# WINDOW #
root = tk.Tk()
SCREEN_WIDTH = root.winfo_screenwidth()-100
SCREEN_HEIGHT = root.winfo_screenheight()-40
root.destroy()

DISPLAYSURF = pygame.display.set_mode((SCREEN_WIDTH,SCREEN_HEIGHT))
DISPLAYSURF.fill(BLACK)
pygame.display.set_caption("Tanner Graph")

######################## 
# CLASSES #
class Rect(pygame.sprite.Sprite):
    def __init__(self,x,y,width,color,value,index):
        super().__init__()
        self.image = pygame.Surface([width, width])
        self.image.fill(color)
        self.rect = self.image.get_rect()
        self.rect.x = x
        self.rect.y = y
        self.maxTime = int(FPS/2)
        self.timer = self.maxTime
        self.value = value
        self.index = index


######################## 
# EVENTS #
def moveMouse():
    global MOUSE_POS
        
    MOUSE_POS = pygame.mouse.get_pos()
    MOUSE_PRESSED = pygame.mouse.get_pressed(num_buttons=3)
    
    # Only run the following code if the left-mouse button has been clicked
    if MOUSE_PRESSED[0]:
        # Only allowed to click on variable nodes, so we only range over them
        for varNode in varNodes:
            # Only interact with those variable nodes that haven't been changed in the past 1/2 second
            if varNode.timer == 0:
                # Only activate for variable nodes intersecting with the mouse when it is pressed
                if varNode.rect.collidepoint(MOUSE_POS):
                    # Reset .timer to .maxTime for updated variable node
                    varNode.timer = varNode.maxTime
                    
                    # Update variable node: Flip the .value and .image.fill color
                    if varNode.value == 0:
                        varNode.value = 1
                        varNode.image.fill(GREEN)
                        # Prep the updateValue for use in the check node update representing the number of adjacent activated variable nodes
                        updateValue = 1
                    else:
                        varNode.value = 0
                        varNode.image.fill(WHITE)
                        updateValue = -1

                    # Update check nodes: Use parityCheckMatrix and .index-es to update adjacent check node .value and .image.fill color
                    for cheNode in cheNodes:
                        if parityCheckMatrix[cheNode.index][varNode.index]:
                            cheNode.value += updateValue
                            if cheNode.value%2:
                                cheNode.image.fill(RED)
                            else:
                                cheNode.image.fill(GREEN)
                            
                            # Redraw the adjusted check node
                            DISPLAYSURF.blit(cheNode.image, cheNode.rect) 

                    # Redraw the adjusted variable node   
                    DISPLAYSURF.blit(varNode.image, varNode.rect)       

# Allows quitting the program using 'x' button
def mouseEvents():
    for event in pygame.event.get():
        if event.type == QUIT:
            pygame.quit()
            sys.exit()

# Decreases timer on recently activated variable nodes
def updateVarNodes():
    for varNode in varNodes:
        if varNode.timer > 0:
            varNode.timer -= 1

######################## 
# CHECK AND VARIABLE NODES #

# Initialize the node groups
cheNodes = pygame.sprite.Group() 
varNodes = pygame.sprite.Group()

# How many of each type of node
numCheNodes = len(parityCheckMatrix)
numVarNodes = len(parityCheckMatrix[0])

# Use to determine appropriate width of node
numNodes = max(numCheNodes,numVarNodes)
nodeSize = min(20,int((SCREEN_WIDTH-numNodes*4)/numNodes))

# Add an appropriate number of variable and check nodes, indexed to rows and columsn in PCM
for row in range(numCheNodes):
    cheNodes.add(Rect(int((row+0.25)*(SCREEN_WIDTH)/numCheNodes),int(SCREEN_HEIGHT/3),nodeSize,GREEN,0,row))
for column in range(numVarNodes):
    varNodes.add(Rect(int((column+0.25)*(SCREEN_WIDTH)/numVarNodes),int(2*SCREEN_HEIGHT/3),nodeSize,WHITE,0,column))

# Initial drawing of the nodes...
for cheNode in cheNodes:
    DISPLAYSURF.blit(cheNode.image, cheNode.rect)
for varNode in varNodes:
    DISPLAYSURF.blit(varNode.image, varNode.rect) 

# ... and edges...
for cheNode in cheNodes:
    for varNode in varNodes:
        if parityCheckMatrix[cheNode.index][varNode.index]:
            pygame.draw.line(DISPLAYSURF,WHITE,cheNode.rect.midbottom,varNode.rect.midtop)

# ... and text
for cheNode in cheNodes:
    text = cheLabels[cheNode.index]
    font_size = font_small.size(text)
    node = (int(cheNode.rect.topleft[0]+(nodeSize-font_size[1])/2),cheNode.rect.topleft[1]-font_size[0]-5)

    text_surface = pygame.transform.rotate(font_small.render(text,False,WHITE),90)
    DISPLAYSURF.blit(text_surface,node)

for varNode in varNodes:
    text = varLabels[varNode.index]
    font_size = font_small.size(text)
    node = (int(varNode.rect.bottomleft[0]+(nodeSize-font_size[1])/2),varNode.rect.bottomleft[1]+5)

    text_surface = pygame.transform.rotate(font_small.render(text,False,WHITE),90)
    DISPLAYSURF.blit(text_surface,node)

# Main loop
while True:     
    updateVarNodes() 
    mouseEvents()
    moveMouse()
        
    pygame.display.update()
    FramePerSec.tick(FPS)